# Business Entity Resolution — Learning Project

Goal: learn ML/DS fundamentals end-to-end by solving a real Entity Resolution problem.

**Pipeline overview:**
1. Load & Explore
2. Understand the "messiness" (name/address noise)
3. Blocking / Candidate Generation
4. Feature Engineering (string similarity)
5. Model Training (binary classifier: match / not match)
6. Validation Split + F0.5 Evaluation
7. Generate `matching_results.tsv` and `candidate_pairs.tsv` for test set

We'll fill in each section together, one at a time. Run cells top to bottom as we go;
don't skip ahead to sections marked TODO until we get there.


## Step 1: Load & Explore

Goal: get a feel for the data before writing a single line of ML code.

In [ ]:
import pandas as pd

# sep="\t" is critical here -- these are TSV files, not CSV.
# If you forget this, pandas will silently load the whole line as ONE column
# and everything downstream will look broken in confusing ways.
s1 = pd.read_csv("dataset/train/train_source1.tsv", sep="\t")
s2 = pd.read_csv("dataset/train/train_source2.tsv", sep="\t")
s3 = pd.read_csv("dataset/train/train_source3.tsv", sep="\t")
gt = pd.read_csv("dataset/train/train_ground_truth.tsv", sep="\t")


### 1.1 Basic shape
`.shape` returns `(num_rows, num_columns)` — tells you scale at a glance.

In [ ]:
print("Source 1 shape:", s1.shape)
print("Source 2 shape:", s2.shape)
print("Source 3 shape:", s3.shape)
print("Ground truth shape:", gt.shape)


### 1.2 Peek at the rows
`.head()` shows the first 5 rows — sanity check that the tab-separated read worked (you should see 4 distinct columns, not 1 giant one).

In [ ]:
s1.head()

In [ ]:
gt.head()

### 1.3 Data types and missing values
`.info()` shows dtype + non-null count per column. Watch for nulls in `business_address` or `country` — missing data changes how we engineer features later.

In [ ]:
s1.info()

### 1.4 Country distribution
`.value_counts()` counts rows per category. Confirm US/India split in train, and remember: test also has France, which we must NOT hardcode against.

In [ ]:
print("Source 1 country counts:")
print(s1["country"].value_counts())
print()
print("Source 2 country counts:")
print(s2["country"].value_counts())


### 1.5 Ground truth shape of the problem
`matched_entity_ids` is a comma-separated string, or empty/NaN for singletons.
Let's count: how many Source-1 entities have ZERO matches (singletons) vs. some matches?

In [ ]:
gt["is_singleton"] = gt["matched_entity_ids"].isna() | (gt["matched_entity_ids"] == "")
print("Singleton count:", gt["is_singleton"].sum())
print("Non-singleton count:", (~gt["is_singleton"]).sum())
print("Singleton fraction:", gt["is_singleton"].mean())


### 1.6 How many matches per non-singleton entity?
We split the comma-separated string into a list and count its length using `.apply()` — runs a function on every row.

In [ ]:
def count_matches(x):
    if pd.isna(x) or x == "":
        return 0
    return len(x.split(","))

gt["num_matches"] = gt["matched_entity_ids"].apply(count_matches)
print(gt["num_matches"].value_counts().sort_index())
print()
print("Max matches for a single entity:", gt["num_matches"].max())


**Pause here.** Run everything above and look at the printed output before moving on — 
we'll discuss what these numbers mean for our strategy before touching Step 2.

## Step 2: Understand the Noise + Clean the Data

Goal: (a) look at real match examples to catalogue noise patterns, (b) quantify how severe
each pattern is, (c) build a first-pass cleaning/normalization function and apply it to all
three sources so later steps (blocking, features) work on consistent text.


### 2.1 Build ID lookups

`.set_index()` turns a column into the DataFrame's index so `.loc[]` lookups by ID are fast
(important once we do this millions of times). We reuse `s1_lookup`, `s2_lookup`, `s3_lookup`,
and `lookup_any()` throughout the rest of the notebook.

In [ ]:
s1_lookup = s1.set_index("entity_id")
s2_lookup = s2.set_index("entity_id")
s3_lookup = s3.set_index("entity_id")

def lookup_any(eid):
    if eid.startswith("S2-"):
        return s2_lookup.loc[eid]
    elif eid.startswith("S3-"):
        return s3_lookup.loc[eid]
    else:
        raise ValueError(f"Unexpected id prefix: {eid}")


### 2.2 Inspect real matched pairs

Pull a handful of non-singleton Source 1 entities and print their matches side by side,
so we see real noise instead of guessing from the problem statement.

In [ ]:
sample_gt = gt[~gt["is_singleton"]].sample(5, random_state=42)

for _, row in sample_gt.iterrows():
    s1_row = s1_lookup.loc[row["source1_entity_id"]]
    print("=" * 80)
    print(f"SOURCE 1: {row['source1_entity_id']}")
    print(f"  name:    {s1_row['business_name']}")
    print(f"  address: {s1_row['business_address']}")
    print(f"  country: {s1_row['country']}")
    print("-" * 80)

    matched_ids = row["matched_entity_ids"].split(",")
    for mid in matched_ids:
        m_row = lookup_any(mid)
        print(f"MATCH: {mid}")
        print(f"  name:    {m_row['business_name']}")
        print(f"  address: {m_row['business_address']}")
        print(f"  country: {m_row['country']}")
    print()


### 2.3 Noise catalogue (from inspection)

**Name noise:**
- Case only: `Davis Family Office` → `DAVIS FAMILY OFFICE`
- Typos: `Office` → `Offie`, `MS` → `M5`
- Punctuation/brackets: `MS Consultancy Corp` → `MS [Consultancy]`
- Legal suffix variants: `Pvt. Ltd.` ↔ `Pvt. Limited`
- Dropped words: `Davis Family Office` → `Davis Family`
- Substituted-but-related words: `MS Consultancy Corp` → `MS Corp Services`
- **Completely unrelated strings (DBA / trade names)**: `Team Air Pvt. Ltd.` → `Mirasol` —
  zero characters in common, still a true match. No name-similarity feature will ever catch this;
  address has to carry the signal here.

**Address noise:**
- Abbreviation swaps: `Circle` ↔ `Cir`, `Tennessee` ↔ `TN`
- Component reordering: `88 Olive Circle, Lebanon, TN` vs `Lebanon, Tennessee, 88 Olive Cir`
- Missing components: house number dropped entirely
- Script/transliteration: Telugu script vs `Telangana` vs `Andhra Pradesh` for the same region
- Junk symbols: `##19821 WHEELWRIGHT DR`

**Implication:** name and address similarity must stay as *separate* features (never collapsed
into one score), since a true match can have near-zero name similarity but near-one address
similarity, or vice versa.


### 2.4 How common is "name is basically useless"?

We saw one dramatic example (`Mirasol`). Before designing blocking around it, we need to know:
is this a rare outlier, or a real chunk of the data? We measure **word overlap** (Jaccard
similarity on the set of words) between Source 1 names and their true matches, across a
larger random sample.

In [ ]:
def word_overlap(name1, name2):
    w1 = set(str(name1).lower().split())
    w2 = set(str(name2).lower().split())
    if not w1 or not w2:
        return 0
    return len(w1 & w2) / len(w1 | w2)

sample_gt2 = gt[~gt["is_singleton"]].sample(2000, random_state=1)

zero_overlap_count = 0
total_pairs = 0
overlaps = []

for _, row in sample_gt2.iterrows():
    s1_name = s1_lookup.loc[row["source1_entity_id"], "business_name"]
    matched_ids = row["matched_entity_ids"].split(",")
    for mid in matched_ids:
        m_name = lookup_any(mid)["business_name"]
        overlap = word_overlap(s1_name, m_name)
        overlaps.append(overlap)
        total_pairs += 1
        if overlap == 0:
            zero_overlap_count += 1

print(f"Total match pairs checked: {total_pairs}")
print(f"Pairs with ZERO word overlap in name: {zero_overlap_count}")
print(f"Fraction with zero overlap: {zero_overlap_count / total_pairs:.4f}")

import pandas as pd
print()
print("Overlap score distribution:")
print(pd.Series(overlaps).describe())


**Read this before moving on:** if the zero-overlap fraction is small (a few percent), name
similarity is still our primary signal and address just rescues the occasional edge case.
If it's large (double digits), address similarity needs to be a first-class signal in blocking
too, not just a tiebreaker fed to the model after name-based blocking already discarded the
true candidate. Paste your numbers back before we design blocking in Step 3.

### 2.5 Cleaning / normalization strategy

Goal: reduce *superficial* noise (case, punctuation, common abbreviations) so that similarity
features in Step 4 aren't fooled by formatting differences that don't reflect a real business
difference. This does **not** solve the DBA/trade-name problem (`Mirasol`) — no amount of
normalization fixes a genuinely different string. That's a matching-model problem, not a
cleaning problem.

What we normalize:
- Lowercase everything
- Strip punctuation (but keep spaces as word separators)
- Collapse repeated whitespace
- Expand common street-type abbreviations (`rd` → `road`, `st` → `street`, etc.)
- Expand common legal-suffix abbreviations in names (`corp` → `corporation`, `ltd` → `limited`,
  `pvt` → `private`, `inc` → `incorporated`, `&` → `and`)
- Strip leading junk characters like `#`

What we deliberately do NOT attempt here (flagging honestly, not hiding it):
- Transliterating non-Latin script (e.g. Telugu) into Latin script — this is a hard NLP problem
  on its own and the rules disallow external APIs/services to do it for us
- Fixing typos automatically (that's what fuzzy/edit-distance similarity in Step 4 is for)
- Resolving DBA/trade names (that's what the *model*, learning from address signal, is for)


In [ ]:
import re

# Street-type abbreviations -> canonical form. Add to this as you discover more from the data.
STREET_ABBREV = {
    r"\brd\b": "road",
    r"\bst\b": "street",
    r"\bave\b": "avenue",
    r"\bdr\b": "drive",
    r"\bcir\b": "circle",
    r"\bblvd\b": "boulevard",
    r"\bln\b": "lane",
    r"\bapt\b": "apartment",
    r"\bbldg\b": "building",
    r"\bfl\b": "floor",
}

# Legal-suffix / business-name abbreviations -> canonical form.
NAME_ABBREV = {
    r"\bcorp\b": "corporation",
    r"\bltd\b": "limited",
    r"\bpvt\b": "private",
    r"\binc\b": "incorporated",
    r"\bllc\b": "llc",     # already a standard token, kept for clarity/no-op
    r"&": " and ",
}

def clean_text(text, abbrev_map):
    if text is None or (isinstance(text, float)):
        return ""
    t = str(text).lower()
    t = t.replace("#", " ")           # strip junk symbols like ##19821
    t = re.sub(r"[^\w\s]", " ", t)    # strip punctuation, keep word chars + spaces
    for pattern, replacement in abbrev_map.items():
        t = re.sub(pattern, replacement, t)
    t = re.sub(r"\s+", " ", t).strip() # collapse repeated whitespace
    return t

def clean_name(name):
    return clean_text(name, NAME_ABBREV)

def clean_address(address):
    return clean_text(address, STREET_ABBREV)


### 2.6 Apply cleaning to all three sources

This adds `business_name_clean` and `business_address_clean` columns to `s1`, `s2`, `s3`.
On ~12M combined rows this can take a couple of minutes with `.apply()` — that's expected;
we're trading a one-time cost now for consistent, reusable clean columns later.

In [ ]:
for df, label in [(s1, "Source 1"), (s2, "Source 2"), (s3, "Source 3")]:
    df["business_name_clean"] = df["business_name"].apply(clean_name)
    df["business_address_clean"] = df["business_address"].apply(clean_address)
    print(f"{label} cleaned.")

s1[["business_name", "business_name_clean", "business_address", "business_address_clean"]].head(10)


### 2.7 Save cleaned data (avoid recomputing every run)

Cleaning ~12M rows takes real time — save the result once so future notebook runs (and later
steps) can just reload it instead of re-cleaning from scratch. Parquet is compact and fast for
this compared to re-parsing TSV + re-cleaning every time.

In [ ]:
import os
os.makedirs("dataset/clean", exist_ok=True)

s1.to_parquet("dataset/clean/s1_clean.parquet")
s2.to_parquet("dataset/clean/s2_clean.parquet")
s3.to_parquet("dataset/clean/s3_clean.parquet")

print("Saved cleaned sources to dataset/clean/")


**If `to_parquet` errors**, you likely need the `pyarrow` package:
run `pip install pyarrow` in your terminal (same environment as your kernel), then re-run the
cell above.

**Pause here.** Run Step 2 top to bottom, paste back:
1. The word-overlap numbers from 2.4
2. A few rows of the cleaned output from 2.6 (does `Cir` become `circle`, does `Corp` become
   `corporation`, etc.?)

Then we'll move to Step 3: designing the actual blocking strategy, informed by what we now
know about how often name alone is enough vs. not.


## Step 3: Blocking / Candidate Generation

Goal: for every Source 1 entity, cheaply produce a *small* set of plausible Source 2 / Source 3
candidates. This sets our **recall ceiling** — a true match that blocking misses can never be
found later, no matter how good the model is.

**Important design decision, before the split:** we hold out a slice of Source 1 entities as a
validation set *now*, before building anything else. This slice's ground truth is never used
for training — it's our only honest way to self-score before submitting to the real leaderboard,
since the actual test set has no labels.

In [ ]:
from sklearn.model_selection import train_test_split

s1_train_ids, s1_val_ids = train_test_split(
    s1["entity_id"], test_size=0.15, random_state=42
)
s1_train_ids = set(s1_train_ids)
s1_val_ids = set(s1_val_ids)

print("Train entities:", len(s1_train_ids))
print("Validation entities:", len(s1_val_ids))


### 3.1 Blocking keys

Two blocking strategies, unioned together (so we catch true matches that either strategy alone
would miss):

- **Name-based key**: first few alphanumeric characters of the cleaned name + country.
  Cheap and effective for most cases, but fails on DBA/trade-name cases like `Mirasol`.
- **Address-based key**: the first digit-token (likely a house/street number) + the first
  non-numeric word of the address + country. This is what rescues cases where the name is
  useless but the address still matches.

Union, not intersection — we want the candidate set to catch either signal. Precision is the
model's job in Step 5; blocking's job is recall.

In [ ]:
import re

def name_block_key(name_clean, k=4):
    letters = re.sub(r"[^a-z0-9]", "", str(name_clean))
    return letters[:k] if letters else "____"

def address_block_key(address_clean):
    tokens = str(address_clean).split()
    number = next((t for t in tokens if t.isdigit()), "")
    words = [t for t in tokens if not t.isdigit()]
    first_word = words[0] if words else ""
    return f"{number}_{first_word}"

for df in [s1, s2, s3]:
    df["name_block_key"] = df["business_name_clean"].apply(name_block_key)
    df["addr_block_key"] = df["business_address_clean"].apply(address_block_key)
    df["country_key"] = df["country"].str.lower()


### 3.2 Generate candidate pairs

Two merges (name-block, address-block) per other-source, unioned and de-duplicated.
**Note:** if a block key group is huge (e.g. a very common 4-letter prefix), the merge for that
group balloons. We check for that below — if you see it, increase `k` in `name_block_key` or
add a third blocking pass.

In [ ]:
def generate_candidates(base_df, other_df):
    m1 = base_df[["entity_id", "country_key", "name_block_key"]].merge(
        other_df[["entity_id", "country_key", "name_block_key"]],
        on=["country_key", "name_block_key"],
        suffixes=("_base", "_other"),
    )[["entity_id_base", "entity_id_other"]]

    m2 = base_df[["entity_id", "country_key", "addr_block_key"]].merge(
        other_df[["entity_id", "country_key", "addr_block_key"]],
        on=["country_key", "addr_block_key"],
        suffixes=("_base", "_other"),
    )[["entity_id_base", "entity_id_other"]]

    pairs = pd.concat([m1, m2], ignore_index=True).drop_duplicates()
    pairs.columns = ["source1_entity_id", "candidate_entity_id"]
    return pairs

# Run on TRAIN ids first (smaller, faster to sanity check before doing the full set)
s1_train_df = s1[s1["entity_id"].isin(s1_train_ids)]

cand_s2 = generate_candidates(s1_train_df, s2)
cand_s3 = generate_candidates(s1_train_df, s3)
candidates_train = pd.concat([cand_s2, cand_s3], ignore_index=True)

print("Train candidate pairs:", len(candidates_train))
print("Candidates per Source 1 entity (avg):",
      len(candidates_train) / s1_train_df["entity_id"].nunique())
print()
print("Largest candidate groups (check for blowup):")
print(candidates_train.groupby("source1_entity_id").size().sort_values(ascending=False).head(10))


### 3.3 Measure blocking recall

This is the number that matters most from this whole step: **what fraction of TRUE matches
(from ground truth) actually made it into our candidate set?** If this is low, no model can fix
it — we need better blocking, not a better classifier.

In [ ]:
gt_train = gt[gt["source1_entity_id"].isin(s1_train_ids)].copy()

cand_lookup = candidates_train.groupby("source1_entity_id")["candidate_entity_id"].apply(set)

def recall_for_row(row):
    if row["is_singleton"]:
        return None  # no true matches to recall; not counted
    true_matches = set(row["matched_entity_ids"].split(","))
    cands = cand_lookup.get(row["source1_entity_id"], set())
    if not true_matches:
        return None
    return len(true_matches & cands) / len(true_matches)

gt_train["blocking_recall"] = gt_train.apply(recall_for_row, axis=1)
overall_recall = gt_train["blocking_recall"].dropna().mean()
perfect_recall_fraction = (gt_train["blocking_recall"].dropna() == 1.0).mean()

print("Mean per-entity blocking recall (non-singletons only):", overall_recall)
print("Fraction of non-singleton entities with FULL recall:", perfect_recall_fraction)


**Read this before continuing.** If mean recall is below ~0.9, stop here and come back to me
with the number — we need to loosen the blocking keys (e.g. shorter name prefix, or add a third
strategy) before building features/model on top of a broken candidate set. Garbage in, garbage
out applies especially hard here, since Step 5's model can only ever be as good as what
blocking handed it.

Once recall looks reasonable, repeat candidate generation for the **validation** ids (so we can
score end-to-end later) and for the **full train set** (needed to build training labels in
Step 5).

In [ ]:
s1_val_df = s1[s1["entity_id"].isin(s1_val_ids)]
cand_s2_val = generate_candidates(s1_val_df, s2)
cand_s3_val = generate_candidates(s1_val_df, s3)
candidates_val = pd.concat([cand_s2_val, cand_s3_val], ignore_index=True)

print("Validation candidate pairs:", len(candidates_val))


## Step 4: Feature Engineering

Goal: turn each (Source1, candidate) pair into numeric similarity features. We keep **name**
and **address** similarity as separate features (never collapsed into one score) — remember the
`Mirasol` case: a true match can have near-zero name similarity but near-one address
similarity.

We use:
- **Word Jaccard** (already built in Step 2) — good for reordering / dropped words
- **Character-level TF-IDF cosine similarity** — robust to typos, partial matches
- **Sequence ratio** (`difflib.SequenceMatcher`) — a stdlib edit-distance-like measure, no extra
  dependency
- **Same-country flag** — cheap but useful signal
- **Length difference** — a weak but free signal (very different lengths are less likely to be
  the same business)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from difflib import SequenceMatcher
import numpy as np

# One vectorizer fit across ALL names from all 3 sources, so vectors are comparable.
# char_wb = character n-grams within word boundaries -- robust to typos and partial words.
all_names = pd.concat([s1["business_name_clean"], s2["business_name_clean"], s3["business_name_clean"]])
name_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2)
name_vectorizer.fit(all_names)

s1_name_vecs = name_vectorizer.transform(s1["business_name_clean"])
s2_name_vecs = name_vectorizer.transform(s2["business_name_clean"])
s3_name_vecs = name_vectorizer.transform(s3["business_name_clean"])

# Row-index maps so we can pull the right vector for any entity_id quickly
s1_row = pd.Series(np.arange(len(s1)), index=s1["entity_id"])
s2_row = pd.Series(np.arange(len(s2)), index=s2["entity_id"])
s3_row = pd.Series(np.arange(len(s3)), index=s3["entity_id"])

def get_vecs(ids, prefix):
    if prefix == "S2":
        return s2_name_vecs[s2_row.loc[ids].values]
    else:
        return s3_name_vecs[s3_row.loc[ids].values]


In [ ]:
def word_jaccard(a, b):
    wa, wb = set(str(a).split()), set(str(b).split())
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

def seq_ratio(a, b):
    return SequenceMatcher(None, str(a), str(b)).ratio()

def build_features(pairs_df):
    """pairs_df: columns source1_entity_id, candidate_entity_id. Returns a feature DataFrame."""
    df = pairs_df.copy()
    df["source"] = df["candidate_entity_id"].str.slice(0, 2)  # 'S2' or 'S3'

    base = s1.set_index("entity_id")
    other_lookup = pd.concat([s2.set_index("entity_id"), s3.set_index("entity_id")])

    b = base.loc[df["source1_entity_id"]].reset_index(drop=True)
    o = other_lookup.loc[df["candidate_entity_id"]].reset_index(drop=True)

    df["same_country"] = (b["country"].values == o["country"].values).astype(int)
    df["name_len_diff"] = (b["business_name_clean"].str.len().values
                            - o["business_name_clean"].str.len().values)
    df["addr_len_diff"] = (b["business_address_clean"].str.len().values
                            - o["business_address_clean"].str.len().values)

    df["name_jaccard"] = [word_jaccard(x, y) for x, y in
                           zip(b["business_name_clean"], o["business_name_clean"])]
    df["addr_jaccard"] = [word_jaccard(x, y) for x, y in
                           zip(b["business_address_clean"], o["business_address_clean"])]

    df["name_seq_ratio"] = [seq_ratio(x, y) for x, y in
                             zip(b["business_name_clean"], o["business_name_clean"])]
    df["addr_seq_ratio"] = [seq_ratio(x, y) for x, y in
                             zip(b["business_address_clean"], o["business_address_clean"])]

    # TF-IDF cosine similarity, computed per source (S2 vs S3 vectors live in different arrays)
    cosine_vals = np.zeros(len(df))
    for src in ["S2", "S3"]:
        mask = (df["source"] == src).values
        if mask.sum() == 0:
            continue
        base_vecs = s1_name_vecs[s1_row.loc[df.loc[mask, "source1_entity_id"]].values]
        cand_vecs = get_vecs(df.loc[mask, "candidate_entity_id"].values, src)
        # vectors are L2-normalized by default -> dot product = cosine similarity
        sims = np.asarray(base_vecs.multiply(cand_vecs).sum(axis=1)).flatten()
        cosine_vals[mask] = sims
    df["name_tfidf_cosine"] = cosine_vals

    return df


**Performance note:** feature building loops in Python per row for the string-similarity
functions, which is slow at full scale (potentially tens of millions of candidate pairs). Before
running this on everything, test on a small slice first:

```python
test_features = build_features(candidates_train.head(2000))
test_features.head()
```

If that looks right and runs fast, then run it on the full `candidates_train` and
`candidates_val` — expect this to take a while (minutes to tens of minutes depending on
candidate volume). Grab a coffee; this is a normal part of working at this scale, not a sign
something's broken.

In [ ]:
train_features = build_features(candidates_train)
val_features = build_features(candidates_val)
print("Train features:", train_features.shape)
print("Val features:", val_features.shape)
train_features.head()


## Step 5: Model Training

Goal: train a binary classifier — is this (Source1, candidate) pair a true match? — using the
features from Step 4.

**Building labels:** a candidate pair is a **positive** (label 1) if it appears in
`train_ground_truth.tsv`'s match list for that Source 1 entity; otherwise it's a **negative**
(label 0). Since candidates vastly outnumber true matches (most candidate pairs are NOT
matches), we downsample negatives for training speed and class balance — this is standard
practice in entity resolution / record linkage, not cutting corners.

We use **LightGBM** (gradient boosted trees): strong on tabular data, fast, interpretable via
feature importance, and MIT-licensed — satisfying the challenge's model-license constraint.

In [ ]:
# pip install lightgbm --break-system-packages   # if not already installed
import lightgbm as lgb

# Build a fast lookup: source1_entity_id -> set of true matched ids (train ids only)
gt_train_lookup = gt_train.set_index("source1_entity_id")["matched_entity_ids"]

def is_true_match(row):
    matches = gt_train_lookup.get(row["source1_entity_id"], "")
    if pd.isna(matches) or matches == "":
        return 0
    return int(row["candidate_entity_id"] in matches.split(","))

train_features["label"] = train_features.apply(is_true_match, axis=1)

positives = train_features[train_features["label"] == 1]
negatives = train_features[train_features["label"] == 0]

print("Positives:", len(positives), "Negatives:", len(negatives))

# Downsample negatives -- keep ~4x as many negatives as positives
neg_sample = negatives.sample(n=min(len(negatives), len(positives) * 4), random_state=42)
train_balanced = pd.concat([positives, neg_sample], ignore_index=True)

feature_cols = ["same_country", "name_len_diff", "addr_len_diff",
                 "name_jaccard", "addr_jaccard", "name_seq_ratio",
                 "addr_seq_ratio", "name_tfidf_cosine"]

X_train = train_balanced[feature_cols]
y_train = train_balanced["label"]

model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42)
model.fit(X_train, y_train)

print("Feature importances:")
for name, imp in sorted(zip(feature_cols, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {name}: {imp}")


**Read the feature importances.** If `name_tfidf_cosine` and `name_jaccard` dominate and
address features barely matter, that's a warning sign given what we learned in Step 2 (the
`Mirasol` case) — it may mean the model is under-using address on the cases where it matters
most. Not necessarily wrong, but worth noting in your methodology writeup.

## Step 6: Validation Split + F0.5 Evaluation

Goal: score the full pipeline (blocking → features → model) on the validation entities we held
out in Step 3 — data the model never trained on — using the exact F0.5 formula from the problem
statement. This is our only honest self-check before submitting.

In [ ]:
val_features["pred_proba"] = model.predict_proba(val_features[feature_cols])[:, 1]

def f_half(precision, recall):
    if precision == 0 and recall == 0:
        return 0.0
    return (1.25 * precision * recall) / (0.25 * precision + recall) if (0.25*precision+recall) > 0 else 0.0

def score_threshold(threshold):
    preds = val_features[val_features["pred_proba"] >= threshold]
    pred_by_entity = preds.groupby("source1_entity_id")["candidate_entity_id"].apply(set)

    gt_val = gt[gt["source1_entity_id"].isin(s1_val_ids)]

    scores = []
    for _, row in gt_val.iterrows():
        true_set = set(row["matched_entity_ids"].split(",")) if not row["is_singleton"] else set()
        pred_set = pred_by_entity.get(row["source1_entity_id"], set())

        if not true_set and not pred_set:
            scores.append(1.0)  # correct singleton
            continue
        if not pred_set:
            scores.append(0.0 if true_set else 1.0)
            continue

        tp = len(true_set & pred_set)
        precision = tp / len(pred_set) if pred_set else 0.0
        recall = tp / len(true_set) if true_set else 0.0
        scores.append(f_half(precision, recall))

    return sum(scores) / len(scores)

# F0.5 is precision-heavy -- sweep thresholds to find what actually maximizes it.
# Don't assume 0.5 is right; that's a common beginner mistake with imbalanced/precision-heavy metrics.
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    print(f"threshold={t:.1f}  F0.5={score_threshold(t):.4f}")


**Pick the threshold with the best F0.5** from the printed sweep above and use it going
forward. Paste back the full sweep table here — the shape of that curve (does it peak sharply,
or plateau?) tells us how sensitive the model is and whether it's worth narrowing the search
further (e.g. checking 0.55, 0.65, etc. around the best point).

## Step 7: Generate Submission Files

Goal: run the exact same pipeline — blocking, features, model, chosen threshold — on the real
**test** set (no ground truth), and produce `candidate_pairs.tsv` and `matching_results.tsv` in
the required format.

**Set `BEST_THRESHOLD` below to whatever came out best in Step 6 before running this.**

In [ ]:
BEST_THRESHOLD = 0.6  # <-- replace with your best threshold from Step 6's sweep

test_s1 = pd.read_csv("dataset/test/test_source1.tsv", sep="\t")
test_s2 = pd.read_csv("dataset/test/test_source2.tsv", sep="\t")
test_s3 = pd.read_csv("dataset/test/test_source3.tsv", sep="\t")

for df in [test_s1, test_s2, test_s3]:
    df["business_name_clean"] = df["business_name"].apply(clean_name)
    df["business_address_clean"] = df["business_address"].apply(clean_address)
    df["name_block_key"] = df["business_name_clean"].apply(name_block_key)
    df["addr_block_key"] = df["business_address_clean"].apply(address_block_key)
    df["country_key"] = df["country"].str.lower()

print("Test sets cleaned:", test_s1.shape, test_s2.shape, test_s3.shape)


In [ ]:
# Re-fit vectorizer vocabulary context to include test names too, so unseen test-only
# character n-grams (e.g. from France) still get sensible vectors. Simpler: transform test
# names using the SAME vectorizer fit earlier -- TfidfVectorizer handles unseen n-grams fine
# (they're just ignored), no need to refit.
test_s1_name_vecs = name_vectorizer.transform(test_s1["business_name_clean"])
test_s2_name_vecs = name_vectorizer.transform(test_s2["business_name_clean"])
test_s3_name_vecs = name_vectorizer.transform(test_s3["business_name_clean"])

test_s1_row = pd.Series(np.arange(len(test_s1)), index=test_s1["entity_id"])
test_s2_row = pd.Series(np.arange(len(test_s2)), index=test_s2["entity_id"])
test_s3_row = pd.Series(np.arange(len(test_s3)), index=test_s3["entity_id"])

cand_test_s2 = generate_candidates(test_s1, test_s2)
cand_test_s3 = generate_candidates(test_s1, test_s3)
candidates_test = pd.concat([cand_test_s2, cand_test_s3], ignore_index=True)
print("Test candidate pairs:", len(candidates_test))


In [ ]:
def build_features_test(pairs_df):
    df = pairs_df.copy()
    df["source"] = df["candidate_entity_id"].str.slice(0, 2)

    base = test_s1.set_index("entity_id")
    other_lookup = pd.concat([test_s2.set_index("entity_id"), test_s3.set_index("entity_id")])

    b = base.loc[df["source1_entity_id"]].reset_index(drop=True)
    o = other_lookup.loc[df["candidate_entity_id"]].reset_index(drop=True)

    df["same_country"] = (b["country"].values == o["country"].values).astype(int)
    df["name_len_diff"] = (b["business_name_clean"].str.len().values
                            - o["business_name_clean"].str.len().values)
    df["addr_len_diff"] = (b["business_address_clean"].str.len().values
                            - o["business_address_clean"].str.len().values)

    df["name_jaccard"] = [word_jaccard(x, y) for x, y in
                           zip(b["business_name_clean"], o["business_name_clean"])]
    df["addr_jaccard"] = [word_jaccard(x, y) for x, y in
                           zip(b["business_address_clean"], o["business_address_clean"])]
    df["name_seq_ratio"] = [seq_ratio(x, y) for x, y in
                             zip(b["business_name_clean"], o["business_name_clean"])]
    df["addr_seq_ratio"] = [seq_ratio(x, y) for x, y in
                             zip(b["business_address_clean"], o["business_address_clean"])]

    cosine_vals = np.zeros(len(df))
    for src, vecs in [("S2", test_s2_name_vecs), ("S3", test_s3_name_vecs)]:
        mask = (df["source"] == src).values
        if mask.sum() == 0:
            continue
        base_vecs = test_s1_name_vecs[test_s1_row.loc[df.loc[mask, "source1_entity_id"]].values]
        row_idx = test_s2_row if src == "S2" else test_s3_row
        cand_vecs = vecs[row_idx.loc[df.loc[mask, "candidate_entity_id"]].values]
        sims = np.asarray(base_vecs.multiply(cand_vecs).sum(axis=1)).flatten()
        cosine_vals[mask] = sims
    df["name_tfidf_cosine"] = cosine_vals

    return df

test_features = build_features_test(candidates_test)
test_features["pred_proba"] = model.predict_proba(test_features[feature_cols])[:, 1]
print(test_features.shape)


### 7.1 Write `candidate_pairs.tsv`

Every candidate we fed the model, grouped by Source 1 entity, per the required format.

In [ ]:
import os
os.makedirs("output", exist_ok=True)

cand_grouped = candidates_test.groupby("source1_entity_id")["candidate_entity_id"].apply(
    lambda ids: ",".join(sorted(set(ids)))
)

cand_out = test_s1[["entity_id"]].rename(columns={"entity_id": "source1_entity_id"}).merge(
    cand_grouped.rename("candidate_entity_ids"), on="source1_entity_id", how="left"
)
cand_out["candidate_entity_ids"] = cand_out["candidate_entity_ids"].fillna("")

cand_out.to_csv("output/candidate_pairs.tsv", sep="\t", index=False)
print("Wrote output/candidate_pairs.tsv:", cand_out.shape)
cand_out.head()


### 7.2 Write `matching_results.tsv`

Only candidates whose predicted probability clears `BEST_THRESHOLD` become final matches.

In [ ]:
final_matches = test_features[test_features["pred_proba"] >= BEST_THRESHOLD]

match_grouped = final_matches.groupby("source1_entity_id")["candidate_entity_id"].apply(
    lambda ids: ",".join(sorted(set(ids)))
)

match_out = test_s1[["entity_id"]].rename(columns={"entity_id": "source1_entity_id"}).merge(
    match_grouped.rename("matched_entity_ids"), on="source1_entity_id", how="left"
)
match_out["matched_entity_ids"] = match_out["matched_entity_ids"].fillna("")

match_out.to_csv("output/matching_results.tsv", sep="\t", index=False)
print("Wrote output/matching_results.tsv:", match_out.shape)
match_out.head()


### 7.3 Validate before submitting

Run the official validator script from your `student_resource/` directory (adjust the path if
yours differs) — **do this in a terminal, not the notebook**, since it's a standalone script:

```
python3 utils/validate_submission.py \
  --matching output/matching_results.tsv \
  --candidate output/candidate_pairs.tsv \
  --test-dir dataset/test
```

It must print `PASS` before you upload anything. If it doesn't, paste the exact numbered issues
it prints here and we'll fix them together — I can't guarantee a first run passes cleanly at
this scale; debugging the validator's specific complaints is a normal, expected part of this
process, not a sign the pipeline is fundamentally broken.